**Mini Projet : Système de Questions-Réponses avec LlamaIndex et HuggingFace**


Ce projet nous guide à travers la création d'un système de questions-réponses basé sur la récupération d'informations, en utilisant LlamaIndex et des modèles HuggingFace open-source. Nous allons charger des documents, les indexer, puis les interroger pour obtenir des réponses structurées.

1. Installer les packages requis


In [1]:
!pip install llama_index llama-index-llms-huggingface llama-index-llms-huggingface-api llama-index-embeddings-huggingface vllm
!pip install accelerate bitsandbytes # Ces packages sont souvent nécessaires pour optimiser l'utilisation des modèles HuggingFace avec le GPU

In [2]:
# 2. Importer les classes nécessaires


from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.storage.storage_context import StorageContext
# Corrected import for load_indices_from_storage
from llama_index.core import load_indices_from_storage # This line is the fix

import torch
import os

In [21]:
# 3. Charger les Documents


folder_path = '/paper'

# Check if the directory exists
if os.path.exists(folder_path):
    print(f"Le dossier '{folder_path}' existe.")
    # List files in the directory
    files_in_folder = os.listdir(folder_path)

    if files_in_folder:
        print(f"Les fichiers suivants ont été trouvés dans '{folder_path}':")
        for file_name in files_in_folder:
            print(file_name)
    else:
        print(f"Le dossier '{folder_path}' est vide.")
else:
    print(f"Le dossier '{folder_path}' n'existe pas.")
    print("Veuillez vous assurer d'avoir créé le dossier 'paper' en utilisant l'explorateur de fichiers de Colab.")

Le dossier '/paper' existe.
Les fichiers suivants ont été trouvés dans '/paper':
A General Language Assistant as a Laboratory for Alignment.pdf
A Cookbook of Self-Supervised Learning.pdf


In [22]:
# 4. Initialiser le Modèle de Langage (LLM)

#Nous allons configurer le Grand Modèle de Langage (LLM) qui sera responsable de la génération des réponses. Ici, nous utilisons TinyLlama/TinyLlama-1.1B-Chat-v1.0.

# Vérification de la disponibilité du GPU pour une exécution plus rapide
if torch.cuda.is_available():
    device = "cuda"
    print("GPU (CUDA) disponible. Le modèle utilisera le GPU.")
else:
    device = "cpu"
    print("GPU (CUDA) non disponible. Le modèle utilisera le CPU, ce qui peut être plus lent.")

# Initialisation du LLM HuggingFace
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

llm = HuggingFaceLLM(
    model=model,
    tokenizer=tokenizer,
    context_window=4096,  # Augmentation de la fenêtre de contexte
    max_new_tokens=256,   # Le nombre maximal de nouveaux tokens que le modèle peut générer pour une réponse.
    # device_map="auto",    # Permet à HuggingFace de choisir automatiquement le meilleur appareil (GPU/CPU).
    # model_kwargs={"torch_dtype": torch.float16} # Décommenter cette ligne si vous avez des problèmes de mémoire avec le GPU, cela utilise un type de donnée moins gourmand.
)

GPU (CUDA) disponible. Le modèle utilisera le GPU.


In [23]:
# 5. Configurer le Modèle d'Embeddings

#Les modèles d'embeddings convertissent le texte de vos documents et de vos requêtes en vecteurs numériques, permettant au système de trouver des informations sémantiquement similaires. Nous utiliserons sentence-transformers/all-MiniLM-L6-v2.

# Initialisation du modèle d'embeddings HuggingFace
embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    device=device # Utilise le même périphérique (GPU ou CPU) que le LLM pour la cohérence.
)

print("Modèle d'embeddings (all-MiniLM-L6-v2) initialisé avec succès.")

Modèle d'embeddings (all-MiniLM-L6-v2) initialisé avec succès.


In [24]:
#6. Appliquer les Modèles aux Paramètres Globaux

# Pour que LlamaIndex sache quels modèles utiliser par défaut pour la génération et les embeddings, nous les définissons dans les paramètres globaux.

Settings.llm = llm
Settings.embed_model = embed_model

print("Les paramètres globaux de LlamaIndex ont été mis à jour avec les modèles LLM et d'embeddings configurés.")


Les paramètres globaux de LlamaIndex ont été mis à jour avec les modèles LLM et d'embeddings configurés.


In [25]:
# 7. Créer l'Index à partir des Documents

# C'est l'étape où LlamaIndex ingère vos documents, les découpe en morceaux, crée des embeddings pour chaque morceau, et construit une structure de données (l'index) qui permet une recherche rapide et efficace.

# Charger les documents depuis le dossier "paper"
print(f"Chargement des documents depuis le dossier '{folder_path}'...")
documents = SimpleDirectoryReader(folder_path).load_data()
print(f"{len(documents)} document(s) chargé(s).")


print("Création de l'index VectorStore à partir des documents chargés...")
index = VectorStoreIndex.from_documents(documents)
print("Index créé avec succès. Vos documents sont maintenant prêts à être interrogés.")

Chargement des documents depuis le dossier '/paper'...
119 document(s) chargé(s).
Création de l'index VectorStore à partir des documents chargés...
Index créé avec succès. Vos documents sont maintenant prêts à être interrogés.


8. Créer le Moteur de Requête et Poser une Question

In [26]:
# Créez un moteur de requête à partir de l'index.
# Cela permet de transformer les questions de l'utilisateur en requêtes que l'index peut comprendre et traiter.
query_engine = index.as_query_engine()

# Posez une question à votre système.
# Remplacez la question ci-dessous par celle que vous souhaitez poser à vos documents.
question = "What is the main topic of the documents?"
print(f"Pose de la question : '{question}'")

# Obtenez la réponse du moteur de requête.
response = query_engine.query(question)

# Affichez la réponse.
print("\nRéponse :")
print(response)

Pose de la question : 'What is the main topic of the documents?'

Réponse :
36

page_label: 1
file_path: /paper/A General Language Assistant as a Laboratory for Alignment.pdf

Contents
1 Introduction 3
1.1 Motivations . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 3
1.2 Research . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 5
1.3 Contributions . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 9
2 Conditioning on Aligned Behavior 9
2.1 Context Distillation . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .


In [27]:
# 8.Index sur le Disque

# Définir le répertoire où l'index sera stocké
PERSIST_DIR = "./storage"

# Créer le répertoire s'il n'existe pas
if not os.path.exists(PERSIST_DIR):
    os.makedirs(PERSIST_DIR)

print(f"Sauvegarde (persistance) de l'index dans le répertoire : {PERSIST_DIR}...")
index.storage_context.persist(persist_dir=PERSIST_DIR)
print("Index persisté avec succès. Vous pouvez maintenant le recharger plus tard.")

Sauvegarde (persistance) de l'index dans le répertoire : ./storage...
Index persisté avec succès. Vous pouvez maintenant le recharger plus tard.


In [30]:
# 9. Interroger l'Index avec des Prompts en Langage Naturel

# Créer un moteur de requête à partir de l'index
query_engine = index.as_query_engine()

# Liste des requêtes d'exemple
queries = [
    "Write a detailed summary of prompting techniques…",
    "What is fine-tuning of language models?",
    "Summarize the sparks of AGI paper…",
    "How can LLMs be used for recommendations in e-commerce?",
    "What are multi-modal embeddings and their applications?"
]

print("\n--- Début des requêtes sur l'index ---")
for i, query in enumerate(queries):
    print(f"\nRequête {i+1}: '{query}'")
    try:
        response = query_engine.query(query)
        print("Réponse du système:")
        print(response)
    except Exception as e:
        print(f"Une erreur est survenue lors de l'exécution de la requête : {e}")

print("\n--- Fin des requêtes ---")


--- Début des requêtes sur l'index ---

Requête 1: 'Write a detailed summary of prompting techniques…'
Réponse du système:

Prompting techniques are a crucial aspect of training large language models (LLMs) for natural language understanding (NLU) and generation (NLG). These techniques involve providing a prompt to the LLM, which is then used to generate a response. Prompting techniques have been studied extensively in the past, and there are several approaches to prompting. In this paper, we compare and contrast several prompting techniques, including:

1. Ethics evaluation from natural language prompts (NEPs)
2. Adversarial honesty evaluation from natural language prompts (AHNs)
3. Toxicity evaluation from natural language prompts (TNLPs)
4. Large language models (LLMs)

NEPs are a type of prompting technique that involves providing a natural language prompt to a LLM, which is then used to generate a response. NEPs have been studied in the past, and there are several approaches to N